# RTMPose Fine-tuning on Google Colab

## Setup
1. Create a zip called `tracking.zip` from the project root:
```bash
cd /path/to/tracking
zip -r ~/Desktop/tracking.zip config.yaml modules/ scripts/ input/RTMPose/ input/labels/ output/extracted_frames/ output/RTMPose/no_weak_20260328_174401/checkpoint_best.pt
```
2. Upload `tracking.zip` to your Google Drive root
3. Enable GPU: **Runtime > Change runtime type > T4 GPU**
4. **Runtime > Run all**

In [ ]:
# CELL 1: Install dependencies
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL,
    )

pip_install(
    "torch", "torchvision", "pyyaml", "tqdm",
    "albumentations", "einops", "timm", "pandas",
    "opencv-python-headless", "Pillow",
)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# CELL 2: Mount Google Drive and extract project
from google.colab import drive
import os, shutil, zipfile

drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/tracking.zip"
EXTRACT_DIR = "/content/tracking"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"tracking.zip not found at {ZIP_PATH}\n"
        "Upload tracking.zip to your Google Drive root first."
    )

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

# Copy zip to local disk first (Drive FUSE can't stream large files)
LOCAL_ZIP = "/tmp/tracking.zip"
print("Copying zip to local disk ...")
shutil.copy2(ZIP_PATH, LOCAL_ZIP)
print(f"Copied ({os.path.getsize(LOCAL_ZIP) / 1e6:.0f} MB)")

os.makedirs(EXTRACT_DIR, exist_ok=True)

print("Extracting tracking.zip ...")
with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)
os.remove(LOCAL_ZIP)
print(f"Extracted to {EXTRACT_DIR}")

os.chdir(EXTRACT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# CELL 3: Convert JSON labels to DLC CSV format
import json
from pathlib import Path
import numpy as np
import pandas as pd

BODYPARTS = [
    "head", "nose", "spine1", "spine2", "spine3", "tailbase",
    "tail1", "tail2", "tail_tip",
    "L_hip", "L_backpaw", "R_backpaw",
    "L_shoulder", "R_frontpaw", "R_shoulder",
    "R_hip", "R_knee", "L_knee", "L_frontpaw",
]
SCORER = "rats"
FIRST_COL_VALUE = "labeled-data"


def load_label_json(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    rows = []
    for item in payload:
        frame_idx = int(item["frame_idx"])
        row = {"frame": frame_idx}
        labels = item.get("labels") or {}
        for kp, val in labels.items():
            vis = int(val[0]) if val else 0
            x = val[1] if len(val) > 1 else None
            y = val[2] if len(val) > 2 else None
            row[f"{kp}_x"] = float(x) if vis and x is not None else float("nan")
            row[f"{kp}_y"] = float(y) if vis and y is not None else float("nan")
        rows.append(row)
    return pd.DataFrame(rows)


def build_dlc_csv(df, video_name, bodyparts, csv_path):
    kp_cols = []
    for bp in bodyparts:
        kp_cols.extend([f"{bp}_x", f"{bp}_y"])

    n = 3 + len(kp_cols)
    h0 = [SCORER] + [""] * (n - 1)
    h1 = ["", "", ""]
    for bp in bodyparts:
        h1.extend([bp, bp])
    h2 = ["", "", ""]
    for _ in bodyparts:
        h2.extend(["x", "y"])

    max_frame = int(pd.to_numeric(df["frame"], errors="coerce").max())
    fw = max(1, len(str(max_frame)))
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    lines = []
    for hdr in [h0, h1, h2]:
        lines.append(",".join(hdr))

    for _, row in df.sort_values("frame").iterrows():
        fi = int(float(row["frame"]))
        parts = [
            FIRST_COL_VALUE,
            video_name,
            f"img{str(fi).zfill(fw)}.png",
        ]
        for c in kp_cols:
            v = row.get(c)
            parts.append("" if pd.isna(v) else str(int(round(float(v)))))
        lines.append(",".join(parts))

    csv_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


labels_dir = Path("input/labels")
out_dir = Path("input/labeled-data")
count = 0
for jp in sorted(labels_dir.glob("*.json")):
    vn = jp.stem
    df = load_label_json(jp)
    csv_path = out_dir / vn / "CollectedData_rats.csv"
    build_dlc_csv(df, vn, BODYPARTS, csv_path)
    count += 1
    print(f"  {jp.name} -> {len(df)} frames")

print(f"\nConverted {count} label files to DLC CSV format.")

In [ ]:
# CELL 4: Prepare training data
prepare_cmd = [
    sys.executable, "scripts/keypoints/RTMPose.py",
    "--model-config", "input/RTMPose/model_rtmpose_x.yaml",
    "--config-overwrite", "input/RTMPose/config_finetune.yaml",
    "--labels-root", "input/labeled-data",
    "--labeled-frames-root", "output/extracted_frames",
    "--workers", "0",
    "prepare",
]

print("Running prepare ...")
result = subprocess.run(prepare_cmd)
if result.returncode != 0:
    raise RuntimeError(f"Prepare failed (exit code {result.returncode})")
print("Prepare complete.")

In [ ]:
# CELL 5: Fine-tune from scratch
train_cmd = [
    sys.executable, "scripts/keypoints/RTMPose.py",
    "--model-config", "input/RTMPose/model_rtmpose_x.yaml",
    "--config-overwrite", "input/RTMPose/config_finetune.yaml",
    "--labels-root", "input/labeled-data",
    "--labeled-frames-root", "output/extracted_frames",
    "--workers", "0",
    "--init-checkpoint",
    "output/RTMPose/no_weak_20260328_174401/checkpoint_best.pt",
    "--save-every-n-epoch", "1",
    "--max-save", "100",
    "train",
    "--prefix", "finetune",
    "--epochs", "100",
    "--selection-metric", "loss",
    "--selection-mode", "min",
]

print("Fine-tuning 100 epochs (selection metric: min val loss) ...")
print("Periodic checkpoints saved every epoch, none pruned.")
print("This will take a while. Watch the output for progress.")
log_path = "/tmp/train.log"
with open(log_path, "w") as log_f:
    result = subprocess.run(train_cmd, stdout=log_f, stderr=subprocess.STDOUT)
if result.returncode != 0:
    print("=== TRAINING OUTPUT (last 3000 chars) ===")
    with open(log_path) as f:
        content = f.read()
        print(content[-3000:] if len(content) > 3000 else content)
    raise RuntimeError(f"Training failed (exit code {result.returncode})")
print("Training complete.")

In [ ]:
# CELL 6: Find best-by-loss checkpoint, evaluate, and save to Drive
import glob, json, math

finetune_runs = sorted(glob.glob("output/RTMPose/finetune_*"))
if not finetune_runs:
    print("No finetune run directory found. Check training output above.")
else:
    latest_run = finetune_runs[-1]
    history_file = f"{latest_run}/history.json"

    # --- Find epoch with lowest val loss ---
    with open(history_file) as f:
        history = json.load(f)

    best_epoch = None
    best_val_loss = float("inf")
    for entry in history:
        val_loss = entry.get("val", {}).get("loss")
        if val_loss is None:
            continue
        try:
            v = float(val_loss)
        except (TypeError, ValueError):
            continue
        if math.isnan(v):
            continue
        if v < best_val_loss:
            best_val_loss = v
            best_epoch = entry["epoch"]

    if best_epoch is None:
        print("WARNING: No valid val loss found in history. Falling back to checkpoint_best.pt")
        best_ckpt = f"{latest_run}/checkpoint_best.pt"
    else:
        best_ckpt = f"{latest_run}/checkpoint_epoch_{best_epoch:03d}.pt"
        print(f"Best val loss: epoch {best_epoch} with val_loss = {best_val_loss:.6f}")

    # --- Evaluate on val ---
    eval_val_cmd = [
        sys.executable, "scripts/keypoints/RTMPose.py",
        "--model-config", "input/RTMPose/model_rtmpose_x.yaml",
        "--config-overwrite", "input/RTMPose/config_finetune.yaml",
        "--labels-root", "input/labeled-data",
        "--labeled-frames-root", "output/extracted_frames",
        "--workers", "0",
        "eval",
        "--checkpoint", best_ckpt,
        "--split", "val",
    ]
    print(f"\nEvaluating best-by-loss checkpoint on val split ...")
    subprocess.run(eval_val_cmd)

    # --- Evaluate on test ---
    eval_test_cmd = eval_val_cmd.copy()
    eval_test_cmd[eval_test_cmd.index("--split") + 1] = "test"
    print(f"\nEvaluating on test split ...")
    subprocess.run(eval_test_cmd)

    # --- Copy checkpoint and history to Drive ---
    drive_ckpt = "/content/drive/MyDrive/checkpoint_best_finetune.pt"
    drive_history = "/content/drive/MyDrive/history.json"
    shutil.copy2(best_ckpt, drive_ckpt)
    shutil.copy2(history_file, drive_history)
    print(f"\nCheckpoint (epoch {best_epoch}, val_loss={best_val_loss:.6f}) saved to:")
    print(f"  Google Drive: checkpoint_best_finetune.pt")
    print(f"History saved to Google Drive: history.json")
    print(f"Run directory: {latest_run}")